# Orbit: Multi-Agent Mission Control on a free Colab T4

This notebook trains `finetune/configs/qlora_small.json` only: Qwen2.5-0.5B-Instruct, QLoRA. It does not use the 1.5B config.

Run the cells from top to bottom. Upload a project zip to `/content/orbit.zip` to train that copy. With no zip, the next cell clones `https://github.com/shmorsi/orbit-multi-agent` (override with the `REPO_URL` environment variable). The install cell must not replace Colab's CUDA torch build. Training writes an adapter and a loss curve. The frozen test is scored once after that.

Do not score the frozen test in a loop.


In [ ]:
import os, shutil, subprocess, zipfile
from pathlib import Path

zip_path = Path('/content/orbit.zip')
if zip_path.exists():
    dest = Path('/content/orbit')
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(dest)
    if not (dest / 'pyproject.toml').exists():
        children = [path for path in dest.iterdir() if path.is_dir() and (path / 'pyproject.toml').exists()]
        if len(children) != 1:
            raise SystemExit('The zip does not contain pyproject.toml at its top level.')
        os.chdir(children[0])
    else:
        os.chdir(dest)
    print('Using the uploaded project.')
else:
    print('No /content/orbit.zip. The next cell clones the default repo.')


In [ ]:
import json, os, shutil, subprocess
from pathlib import Path

repo = os.environ.get('REPO_URL', 'https://github.com/shmorsi/orbit-multi-agent').strip()
if Path('pyproject.toml').exists():
    print('Project already present. Skipping clone.')
else:
    target = Path('/content/orbit-repo')
    if target.exists():
        shutil.rmtree(target)
    subprocess.check_call(['git', 'clone', repo, str(target)])
    os.chdir(target)
    print('cloned', repo)
print(subprocess.check_output(['nvidia-smi'], text=True))
config = json.loads(Path('finetune/configs/qlora_small.json').read_text())
assert config['base_model'] == 'Qwen/Qwen2.5-0.5B-Instruct'
assert config['method'] == 'qlora'
print('project', Path.cwd())
print('base model', config['base_model'])


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade-strategy', 'only-if-needed', '-r', 'finetune/requirements-colab.txt'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'])
# Subprocess import: Colab's kernel puts /content on sys.path and can load the unzipped folder as a namespace package.
print(subprocess.check_output([sys.executable, '-c', 'import orbit_agents, os; print(orbit_agents.__version__, os.path.dirname(orbit_agents.__file__))'], text=True))
subprocess.check_call([sys.executable, 'finetune/leakage.py'])


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, 'finetune/train.py', '--config', 'finetune/configs/qlora_small.json', '--output', 'finetune/outputs'])


In [ ]:
from pathlib import Path
path = Path('finetune/outputs/loss_curve.png')
if not path.exists():
    raise SystemExit('Training did not write a loss curve. There is no score to invent.')
from IPython.display import Image, display
display(Image(filename=str(path)))
print(Path('finetune/outputs/RUN.json').read_text())


In [ ]:
import subprocess, sys
commands = [
    [sys.executable, '-m', 'orbit_agents.eval.run', '--split', 'test', '--mode', 'single', '--backend', 'transformers', '--model', 'Qwen/Qwen2.5-0.5B-Instruct', '--out', 'results/single_base.json'],
    [sys.executable, '-m', 'orbit_agents.eval.run', '--split', 'test', '--mode', 'team', '--backend', 'transformers', '--model', 'Qwen/Qwen2.5-0.5B-Instruct', '--out', 'results/team_base.json'],
    [sys.executable, '-m', 'orbit_agents.eval.run', '--split', 'test', '--mode', 'team', '--backend', 'transformers', '--model', 'Qwen/Qwen2.5-0.5B-Instruct', '--adapter', 'finetune/outputs/adapter', '--out', 'results/team_lora.json'],
    [sys.executable, '-m', 'orbit_agents.eval.compare', '--single', 'results/single_base.json', '--team', 'results/team_base.json', '--lora', 'results/team_lora.json', '--out', 'results/comparison.json'],
]
for command in commands:
    subprocess.check_call(command)


In [ ]:
import shutil
from pathlib import Path
bundle = Path('/content/orbit-outputs')
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir()
paths = [
    Path('finetune/outputs/adapter'),
    Path('finetune/outputs/loss_curve.png'),
    Path('finetune/outputs/RUN.json'),
    Path('results/single_base.json'),
    Path('results/team_base.json'),
    Path('results/team_lora.json'),
    Path('results/comparison.json'),
]
for path in paths:
    if not path.exists():
        raise SystemExit(f'missing {path}')
    target = bundle / path.name
    if path.is_dir():
        shutil.copytree(path, target)
    else:
        shutil.copy2(path, target)
archive = shutil.make_archive('/content/outputs', 'zip', bundle)
print(archive)
try:
    from google.colab import files
    files.download('/content/outputs.zip')
except Exception as exc:
    print('Download skipped outside Colab:', type(exc).__name__)


Do not score the frozen test in a loop. Download `/content/outputs.zip` before the runtime ends. The adapter is not a task-success number; the comparison file is.
